In [37]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

from surprise import Dataset, Reader, SVD, accuracy, dump
from surprise.model_selection import GridSearchCV
from collections import defaultdict

import json
import pickle


### Environment Setup & Data Pipelines 
This section sets up our data pipeline and creates a strict wall between our training data and our testing data. To save computer memory when dealing with millions of Amazon reviews, we drop the timestamps immediately upon loading. We keep our data clean by splitting the dataset into training (80%) and testing (20%) sets before doing any filtering or calculations. This ensures our final test results reflect how the model handles completely unseen data. Once the split is safe, this section filters out low-volume items and users with fewer than fifty reviews, using a combined filter to keep the dataset stable and clean.


In [2]:
df_raw = pd.read_csv("/Users/pc/Downloads/ratings_Electronics (1).csv")
df_raw.columns = ["user_id", "product_id", "rating", "timestamp"]

In [3]:
df_raw.info()
df_raw.isna().sum()
df_raw.duplicated().sum()
(~df_raw["rating"].between(0, 5)).sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7824481 entries, 0 to 7824480
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   user_id     object 
 1   product_id  object 
 2   rating      float64
 3   timestamp   int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 238.8+ MB


0

In [4]:
df_raw = df_raw.drop(columns=["timestamp"])
df_train_raw, df_test = train_test_split(df_raw, test_size=0.20, random_state=42)


In [5]:
user_counts = df_train_raw["user_id"].value_counts()
item_counts = df_train_raw["product_id"].value_counts()
keep_users = user_counts[user_counts >= 50].index
keep_items = item_counts[item_counts >= 50].index

df_train = df_train_raw[df_train_raw["user_id"].isin(keep_users) & df_train_raw["product_id"].
                        isin(keep_items)].copy()

### The "Cold-Start" Popularity Engine
This section builds a reliable popularity framework to recommend products to brand-new visitors who do not have a purchase history yet. In online shopping, raw averages can be misleading; a niche product with a single five-star review should not be ranked above a popular product with thousands of highly rated reviews. To fix this, this section uses the classic IMDb Bayesian Weighted Rating formula. This math pulls low-volume items safely toward the global platform average while allowing high-volume items to rely on their true score, creating a fair ranking system that new users can trust.


In [6]:
C = df_train["rating"].mean()
m = df_train["product_id"].value_counts().quantile(0.90)
product_stats = df_train.groupby("product_id").agg(
    v=("rating", "count"),
    R=("rating", "mean")
)

In [7]:
qualified_products = product_stats[product_stats["v"] >= m].copy()

In [8]:
def weighted_rating(x, m=m, C=C):
    v = x["v"]
    R = x["R"]
    return (v / (v + m) * R) + (m / (v + m) * C)

In [9]:
qualified_products["weighted_score"] = qualified_products.apply(weighted_rating, axis=1)
top_recommendations = qualified_products.sort_values(by="weighted_score", ascending=False)

### Memory-Optimised Item-to-Item Similarity
This block sets up a real-time cross-selling system to power "Frequently Bought Together" carousels while keeping a low memory footprint. Standard pivot tables waste massive amounts of computer RAM by storing millions of empty zeros for items users have never clicked on, which easily crashes servers. To make this code scale efficiently, this section maps alphanumeric Amazon IDs into simple numbers and passes a Compressed Sparse Row (CSR) matrix to Scikit-Learn. This calculates product similarities quickly while using only a fraction of the hardware memory.


In [10]:
top_100_items = df_train["product_id"].value_counts().head(100).index
df_sample = df_train[df_train["product_id"].isin(top_100_items)].copy()


In [11]:
df_sample['user_code'] = df_sample['user_id'].astype('category').cat.codes
df_sample['product_code'] = df_sample['product_id'].astype('category').cat.codes

In [12]:
sparse_matrix = csr_matrix(
    (df_sample['rating'], (df_sample['product_code'], df_sample['user_code']))
)

In [13]:
item_similarity = cosine_similarity(sparse_matrix)
unique_items = df_sample['product_id'].astype('category').cat.categories
item_sim_df = pd.DataFrame(item_similarity, index=unique_items, columns=unique_items)


In [14]:
sample_item = item_sim_df.index[0]
similar_items = item_sim_df[sample_item].sort_values(ascending=False)

In [15]:
print(f"Generated similarity grid size: {item_sim_df.shape}")
print(f"Top 3 behavioural matches for Product {sample_item}:")
print(similar_items.head(3))

Generated similarity grid size: (100, 100)
Top 3 behavioural matches for Product B00004ZCJE:
B00004ZCJE    1.000000
B006W8U2MU    0.143323
B000VX6XL6    0.139349
Name: B00004ZCJE, dtype: float64


### Tuning Engine & Model Selection (GridSearchCV)
This part of the code handles the personalised core of the system, using a Singular Value Decomposition (SVD) matrix factorisation algorithm to uncover hidden user shopping habits. Instead of guessing model settings blindly, this section uses an automated grid search tool to test eight different model variations. By running a 3-Fold Cross-Validation loop on our training data, the computer runs twenty-four mini-tests to find the exact factor sizes, learning speeds, and safety penalties that produce the lowest error. The winning model configuration is then trained on the full training pool so it can learn from all available data.


In [17]:
reader = Reader(rating_scale=(1.0, 5.0))
surp_cv_data = Dataset.load_from_df(df_train[["user_id", "product_id", "rating"]], reader)

In [19]:
param_grid = {
    "n_factors": [10, 20],
    "lr_all": [0.005, 0.01],
    "reg_all": [0.02, 0.1]
}

In [20]:
gs = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=-1)
gs.fit(surp_cv_data)


In [21]:
best_rmse_score = gs.best_score["rmse"]
best_hyperparams = gs.best_params["rmse"]

In [22]:
svd_model = gs.best_estimator["rmse"]
full_trainset = surp_cv_data.build_full_trainset()
svd_model.fit(full_trainset)

### System Evaluation — Validation Audit
Now that our machine learning core is trained, this section runs a final quality check using the untouched 20% test split we hid away in Section 1. While the grid search loop in Section 4 acted as an internal audition, this section serves as the model's final exam against completely new data. We feed this hidden data into our tuned SVD model and score the predictions using the native accuracy toolkit from the Surprise library, outputting a stable and realistic final Root Mean Squared Error (RMSE) of 1.4049.


In [29]:
surp_test = Dataset.load_from_df(df_test[["user_id", "product_id", "rating"]], reader)
testset = surp_test.build_full_trainset().build_testset()

In [30]:
predictions = svd_model.test(testset)

In [34]:
final_test_rmse = accuracy.rmse(predictions)


RMSE: 1.4049


###  Production Artifact Export
A live web server cannot afford to clean raw data or re-train a heavy machine learning model every time a customer refreshes a web page, making file saving essential for real-world deployment. This section saves our work into three specific files: a tiny text JSON array for instant trending homepages, a pickled binary of our cross-selling matrix for fast product page lookups, and a frozen SVD model file for real-time personalised predictions.


In [38]:
cold_start_items = top_recommendations.index.tolist()[:10]
with open("cold_start_recommendations.json", "w") as f:
    json.dump(cold_start_items, f)

In [39]:
with open("item_similarity.pkl", "wb") as f:
    pickle.dump(item_sim_df, f)

In [40]:
model_filename = "svd_model.pkl"
dump.dump(model_filename, algo=svd_model)